# ParaKoop — DrivAerNet OpenFOAM Validation

Runs 3–5 real DrivAerNet car STL files through OpenFOAM simpleFoam (k-ω SST RANS)
and compares CFD Cd against:
1. DrivAerNet ground-truth Cd (from `geometry_features.csv`)
2. ParaKoop model predictions

**Runtime**: ~1–2 hrs per case on Colab CPU (GPU not used by OpenFOAM)

**Prerequisites**:
- Mount Google Drive containing `parakoop/` repo and DrivAerNet STL zips
- Colab compute credits (High-RAM recommended)

## 1. Mount Drive & Clone/Pull Repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Clone latest code from GitHub
if not os.path.exists('/content/parakoop'):
    !git clone https://github.com/Ranaam21/Parakoop.git /content/parakoop
else:
    !git -C /content/parakoop pull origin main

PARAKOOP_DIR = '/content/parakoop'

# Data and checkpoint stay in Drive (not tracked by git)
DATA_DIR = '/content/drive/MyDrive/Car_CFD/parakoop/data/drivaernet'
CKPT     = '/content/drive/MyDrive/Car_CFD/parakoop/checkpoints/unified/parakoop_unified_best.pt'

os.chdir(PARAKOOP_DIR)
print('Repo ready :', PARAKOOP_DIR)
print('Data dir   :', DATA_DIR)
print('Checkpoint :', CKPT)

## 2. Install OpenFOAM v2312

In [ ]:
%%bash
# Add OpenFOAM repository and install
curl -s https://dl.openfoam.com/add-debian-repo.sh | sudo bash
sudo apt-get update -qq
sudo apt-get install -y openfoam2312 2>&1 | tail -5
echo 'OpenFOAM installed'

In [ ]:
# Source OpenFOAM environment for all subsequent bash cells
import subprocess
OF_BASHRC = '/usr/lib/openfoam/openfoam2312/etc/bashrc'
result = subprocess.run(['bash', '-c', f'source {OF_BASHRC} && env'],
                        capture_output=True, text=True)
for line in result.stdout.split('\n'):
    if line.startswith(('FOAM_', 'WM_', 'PATH=', 'LD_LIBRARY')):
        key, _, val = line.partition('=')
        os.environ[key] = val
print('OpenFOAM env loaded. Version:', os.environ.get('WM_PROJECT_VERSION', 'unknown'))

## 3. Install Python Dependencies & Load Model

In [ ]:
!pip install -q torch numpy pandas scikit-learn scipy tqdm

import sys
sys.path.insert(0, PARAKOOP_DIR)

import torch
import pandas as pd
import numpy as np
from koopman.model import ParaKoopModel
from data_pipeline.unified_loader import load_unified, THETA_COLS

# CKPT defined in Cell 1 — reads from Drive
ckpt  = torch.load(CKPT, map_location='cpu', weights_only=False)
cfg   = ckpt.get('model_cfg', {})
model = ParaKoopModel(
    phi_dim=cfg.get('phi_dim', 1),
    theta_dim=cfg.get('theta_dim', 8),
    koopman_dim=cfg.get('koopman_dim', 128),
    operator_rank=cfg.get('operator_rank', 16),
    hidden_lift=cfg.get('hidden_lift', 64),
    hidden_op=cfg.get('hidden_op', 64),
    lambda_fp_cd=cfg.get('lambda_fp_cd', 0.1),
)
model.load_state_dict(ckpt['model_state'])
model.eval()
print(f'Model loaded  : {sum(p.numel() for p in model.parameters()):,} params')
print(f'Checkpoint    : {CKPT}')

## 4. Select DrivAerNet STL Cases

In [ ]:
import zipfile, shutil

# Load geometry features (contains ground-truth Cd from DrivAerNet CFD)
geo_df = pd.read_csv(os.path.join(DATA_DIR, 'geometry_features.csv'))
cd_csv = pd.read_csv(os.path.join(DATA_DIR, 'DrivAerNetPlusPlus_Cd_8k_Updated.csv'))

# Pick one representative design per body style (lowest-Cd for clean flow)
CASES = [
    {'zip': 'E_S_WW_WM.zip',   'design_id': None, 'style': 'estateback'},
    {'zip': 'F_S_WWC_WM.zip',  'design_id': None, 'style': 'fastback'},
    {'zip': 'N_S_WWC_WM.zip',  'design_id': None, 'style': 'notchback'},
]

# Auto-select design_id: pick the one with median Cd for each style
for c in CASES:
    config_prefix = c['zip'].replace('.zip', '')
    subset = geo_df[geo_df['config'] == config_prefix].copy()
    if len(subset) == 0:
        # Try matching by design_id prefix
        prefix = config_prefix.split('_')[0] + '_' + config_prefix.split('_')[1]
        subset = geo_df[geo_df['design_id'].str.startswith(prefix)].copy()
    median_cd = subset['cd'].median()
    closest = subset.iloc[(subset['cd'] - median_cd).abs().argsort()[:1]]
    c['design_id'] = closest['design_id'].values[0]
    c['cd_csv']    = float(closest['cd'].values[0])
    c['geo']       = closest.iloc[0].to_dict()
    print(f"{c['style']:12s}: {c['design_id']}  Cd_csv={c['cd_csv']:.4f}")

In [ ]:
# Extract STL files for selected designs
STL_DIR = '/content/drivaernet_stls'
os.makedirs(STL_DIR, exist_ok=True)

for c in CASES:
    zip_path = os.path.join(DATA_DIR, 'meshes', c['zip'])
    stl_name = c['design_id'] + '.stl'
    out_path = os.path.join(STL_DIR, stl_name)
    if not os.path.exists(out_path):
        print(f"Extracting {stl_name} from {c['zip']}...")
        with zipfile.ZipFile(zip_path) as z:
            # Find matching file in zip
            matches = [n for n in z.namelist() if c['design_id'] in n]
            if matches:
                z.extract(matches[0], STL_DIR)
                extracted = os.path.join(STL_DIR, matches[0])
                if extracted != out_path:
                    shutil.move(extracted, out_path)
                print(f'  → {out_path}  ({os.path.getsize(out_path)/1e6:.0f}MB)')
            else:
                print(f'  WARNING: {stl_name} not found in {c["zip"]}')
    else:
        print(f'{stl_name} already extracted')
    c['stl_path'] = out_path

## 5. Generate & Run OpenFOAM Cases

In [ ]:
import sys
sys.path.insert(0, PARAKOOP_DIR)
from openfoam.drivaernet_case_generator import generate_drivaernet_case

CASES_DIR = '/content/of_cases'
os.makedirs(CASES_DIR, exist_ok=True)

for c in CASES:
    case_dir = os.path.join(CASES_DIR, c['design_id'])
    generate_drivaernet_case(
        stl_path   = c['stl_path'],
        geo        = c['geo'],
        output_dir = case_dir,
    )
    c['case_dir'] = case_dir
    print(f"Case ready: {case_dir}")

In [ ]:
import subprocess, time

def run_openfoam_case(case_dir, n_procs=8):
    """Run blockMesh → snappyHexMesh → decomposePar → simpleFoam in parallel."""
    OF_ENV = f'source /usr/lib/openfoam/openfoam2312/etc/bashrc && '
    steps = [
        ('clean',               f'find constant/polyMesh -maxdepth 1 -type f ! -name blockMeshDict -delete 2>/dev/null; true'),
        ('blockMesh',           'blockMesh'),
        ('surfaceFeatureExtract','surfaceFeatureExtract'),
        ('snappyHexMesh',       'snappyHexMesh -overwrite'),
        ('decomposePar',        f'decomposePar -force'),
        ('simpleFoam',          f'mpirun -np {n_procs} simpleFoam -parallel'),
        ('reconstructPar',      'reconstructPar -latestTime'),
    ]
    for name, cmd in steps:
        print(f'  [{name}]', end=' ', flush=True)
        t0 = time.time()
        result = subprocess.run(
            f'cd {case_dir} && {OF_ENV} {cmd} > log.{name} 2>&1',
            shell=True, executable='/bin/bash'
        )
        elapsed = time.time() - t0
        if result.returncode != 0:
            print(f'FAILED ({elapsed:.0f}s)')
            subprocess.run(f'tail -10 {case_dir}/log.{name}', shell=True)
            return False
        print(f'done ({elapsed:.0f}s)')
    return True

for c in CASES:
    print(f"\n{'─'*60}")
    print(f"Running: {c['design_id']}  ({c['style']})")
    c['foam_ok'] = run_openfoam_case(c['case_dir'], n_procs=8)

## 6. Extract CFD Results & Compare

In [ ]:
import glob
from data_pipeline.stl_features import extract_geometry_features
from data_pipeline.unified_loader import THETA_MIN, THETA_MAX

def parse_cd_cl(case_dir, n_avg=100):
    """Read time-averaged Cd/Cl from forceCoeffs output."""
    patterns = [
        os.path.join(case_dir, 'postProcessing', '**', 'coefficient.dat'),
        os.path.join(case_dir, 'postProcessing', '**', 'forceCoeffs.dat'),
    ]
    for pat in patterns:
        files = glob.glob(pat, recursive=True)
        if files:
            rows, cd_col, cl_col = [], None, None
            with open(files[0]) as f:
                for line in f:
                    line = line.strip()
                    if line.startswith('#'):
                        parts = line.lstrip('#').split()
                        if 'Cd' in parts: cd_col = parts.index('Cd')
                        if 'Cl' in parts and cl_col is None: cl_col = parts.index('Cl')
                        continue
                    vals = line.split()
                    if len(vals) >= 3:
                        try: rows.append([float(v) for v in vals])
                        except: pass
            if rows:
                arr = np.array(rows)
                tail = arr[-n_avg:]
                cd_col = cd_col or 1
                cl_col = cl_col or 4
                return float(np.mean(tail[:, cd_col])), float(np.mean(tail[:, cl_col]))
    return None, None

def model_predict(design_id, geo, dataset):
    """Get ParaKoop Cd prediction for a DrivAerNet design."""
    theta_all, cd_all, _, _ = dataset.get_arrays()
    # Find the design in the dataset by design_id
    idx = dataset.design_ids.index(design_id) if hasattr(dataset, 'design_ids') else None
    if idx is not None:
        t = torch.tensor(theta_all[idx], dtype=torch.float32).unsqueeze(0)
    else:
        # Fall back: build theta from geo dict
        from data_pipeline.unified_loader import _drivaernet_row_to_theta
        import pandas as pd
        t = torch.tensor(_drivaernet_row_to_theta(pd.Series(geo)),
                         dtype=torch.float32).unsqueeze(0)
    with torch.no_grad():
        cd_p, cl_p, _ = model.forward_cd_only(t)
    return float(cd_p), float(cl_p)

dataset = load_unified(verbose=False)

records = []
for c in CASES:
    cd_cfd, cl_cfd = (parse_cd_cl(c['case_dir']) if c.get('foam_ok') else (None, None))
    cd_pk, cl_pk   = model_predict(c['design_id'], c['geo'], dataset)
    records.append({
        'design_id': c['design_id'], 'style': c['style'],
        'cd_csv': c['cd_csv'],
        'cd_cfd': round(cd_cfd, 4) if cd_cfd else None,
        'cd_pk':  round(cd_pk, 4),
        'cl_cfd': round(cl_cfd, 4) if cl_cfd else None,
        'cl_pk':  round(cl_pk, 4),
    })

df = pd.DataFrame(records)
df['err_csv_vs_pk']  = (df['cd_csv']  - df['cd_pk']).abs()
df['err_cfd_vs_pk']  = (df['cd_cfd']  - df['cd_pk']).abs()
df['err_cfd_vs_csv'] = (df['cd_cfd']  - df['cd_csv']).abs()
print(df[['design_id','style','cd_csv','cd_cfd','cd_pk',
           'err_csv_vs_pk','err_cfd_vs_pk','err_cfd_vs_csv']].to_string(index=False))
df.to_csv(os.path.join(PARAKOOP_DIR, 'results/drivaernet_cfd_validation.csv'), index=False)

## 7. Summary Table

In [ ]:
print('\n' + '═'*80)
print('  ParaKoop — DrivAerNet CFD Validation  (simpleFoam k-ω SST, U∞=40m/s)')
print('═'*80)
print(f"{'Design':<25} {'Style':<12} {'Cd_csv':>8} {'Cd_cfd':>8} {'Cd_PK':>8} "
      f"{'|Δ|csv':>8} {'|Δ|cfd':>8}")
print('  ' + '─'*74)
for _, r in df.iterrows():
    cfd_s   = f"{r.cd_cfd:.4f}" if r.cd_cfd else '  n/a  '
    ecfd_s  = f"{r.err_cfd_vs_pk:.4f}" if r.cd_cfd else '  n/a  '
    print(f"  {r.design_id:<23} {r.style:<12} {r.cd_csv:>8.4f} "
          f"{cfd_s:>8} {r.cd_pk:>8.4f} {r.err_csv_vs_pk:>8.4f} {ecfd_s:>8}")
print('  ' + '─'*74)
valid = df.dropna(subset=['cd_cfd'])
if len(valid):
    print(f"  Mean |Δ| PK vs CSV : {df.err_csv_vs_pk.mean():.4f}")
    print(f"  Mean |Δ| PK vs CFD : {valid.err_cfd_vs_pk.mean():.4f}")
    print(f"  Mean |Δ| CFD vs CSV: {valid.err_cfd_vs_csv.mean():.4f}  (CFD setup accuracy check)")
print('═'*80)